# Pipeline 01: Text-to-Parameters (Editing Brain)

**Goal:** Convert natural language editing instructions into structured JSON parameters. This is the "brain" that understands what you want and translates it into precise, executable adjustments.

**Architecture:**
- Input: text prompt ("make the top right warmer, brighten the face, add film grain")
- Output: JSON with global adjustments, tone curves, color mixer, and regional targets
- Implementation: Claude API (future: distilled local Qwen model)

**Why this approach:**
- Separates "understanding" from "executing" — parameters are interpretable
- No diffusion artifacts — every edit is a mathematical pixel transform
- Every (prompt, params) pair becomes training data for the future local model

In [1]:
# Setup — mount Drive, install deps, import shared utilities
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/photo-style-rl'

!pip install anthropic pillow numpy matplotlib opencv-python-headless -q

# Copy shared module to working directory so we can import it
import shutil
shutil.copy(f'{PROJECT}/src/shared.py', '/content/shared.py')

import json
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
from shared import (
    PROJECT, RAW_DIR, EDITED_DIR, CHECKPOINTS_DIR,
    pair_files, get_number, image_to_base64, extract_json,
    DeterministicRenderer, show_before_after
)

print("Setup complete")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 15.6 MB/s eta 0:00:00
Setup complete


In [2]:
# Developer reference: full parameter schema with ranges and descriptions.
# NOTE: This dict is not used at runtime — it's documentation only.
# The authoritative schema lives in shared.py (VALID_BASIC_KEYS, VALID_TONE_CURVE_KEYS,
# VALID_COLOR_NAMES) and is enforced by validate_payload() during training data generation.
# If you add a new renderer parameter, update shared.py first, then mirror it here.

EDIT_SCHEMA = {
    "global": {
        "exposure":    {"range": [-5.0, 5.0],  "desc": "Overall brightness in stops"},
        "contrast":    {"range": [-100, 100],   "desc": "Midtone separation"},
        "highlights":  {"range": [-100, 100],   "desc": "Bright area recovery (-) or boost (+)"},
        "shadows":     {"range": [-100, 100],   "desc": "Dark area lift (+) or crush (-)"},
        "whites":      {"range": [-100, 100],   "desc": "White point clipping"},
        "blacks":      {"range": [-100, 100],   "desc": "Black point clipping"},
        "temperature": {"range": [-100, 100],   "desc": "Cool/blue (-) to warm/yellow (+)"},
        "tint":        {"range": [-100, 100],   "desc": "Green (-) to magenta (+)"},
        "texture":     {"range": [-100, 100],   "desc": "Medium-frequency detail"},
        "clarity":     {"range": [-100, 100],   "desc": "Midtone local contrast"},
        "dehaze":      {"range": [-100, 100],   "desc": "Cut haze (+) or add atmosphere (-)"},
        "vibrance":    {"range": [-100, 100],   "desc": "Smart saturation (protects skin)"},
        "saturation":  {"range": [-100, 100],   "desc": "Uniform color intensity"},
    },
    "tone_curve": {
        "blacks":      {"range": [-100, 100]},
        "shadows":     {"range": [-100, 100]},
        "midtones":    {"range": [-100, 100]},
        "highlights":  {"range": [-100, 100]},
        "whites":      {"range": [-100, 100]},
    },
    "color_mixer": {
        "desc": "8 color channels × hue/saturation/luminance shifts",
        "colors": ["reds", "oranges", "yellows", "greens", "aquas", "blues", "purples", "magentas"],
        "per_color": {"h": {"range": [-180, 180]}, "s": {"range": [-100, 100]}, "l": {"range": [-100, 100]}},
    },
}

print("Parameter schema loaded (developer reference — not used at runtime)")
print(f"  Global: {len(EDIT_SCHEMA['global'])} parameters")
print(f"  Tone curve: 5 bands (blacks → whites)")
print(f"  Color mixer: 8 channels × 3 axes (h/s/l)")


Parameter schema loaded (developer reference — not used at runtime)
  Global: 13 parameters
  Tone curve: 5 bands (blacks → whites)
  Color mixer: 8 channels × 3 axes (h/s/l)


## 1. LLM-Based Parameter Generation

Claude interprets natural language editing instructions and outputs structured JSON. This serves two purposes:
1. **Immediate functionality** — the system works now, with high-quality text understanding
2. **Training data** — every (prompt, params) pair becomes training data for a local model (Pipeline 06-07)

In [3]:
import anthropic
from google.colab import userdata

api_key = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=api_key)

# The system prompt uses the flat, region-keyed format that matches:
#   - NB06 synthetic data generation
#   - NB07 Qwen fine-tuning target
#   - NB04/05 text override parsing
# Top-level keys are region names (global, sky, face, subject, background, etc.).
# tone_curve and color_mixer are nested inside each region dict — not top-level siblings.
SYSTEM_PROMPT = """You are an expert photo editor and colorist. Convert natural language editing instructions into precise numeric parameters.

Output ONLY valid JSON using this flat, region-keyed format. Include only parameters that differ from default (0). Omit defaults.

{
    "global": {
        "exposure": <-5.0 to 5.0>,
        "contrast": <-100 to 100>,
        "highlights": <-100 to 100>,
        "shadows": <-100 to 100>,
        "whites": <-100 to 100>,
        "blacks": <-100 to 100>,
        "temperature": <-100 to 100>,
        "tint": <-100 to 100>,
        "texture": <-100 to 100>,
        "clarity": <-100 to 100>,
        "dehaze": <-100 to 100>,
        "vibrance": <-100 to 100>,
        "saturation": <-100 to 100>,
        "tone_curve": {
            "blacks": <-100 to 100>, "shadows": <-100 to 100>,
            "midtones": <-100 to 100>, "highlights": <-100 to 100>, "whites": <-100 to 100>
        },
        "color_mixer": {
            "<color>": {"h": <-180 to 180>, "s": <-100 to 100>, "l": <-100 to 100>}
        }
    },
    "<region>": {
        <same keys as global — only include if edit is region-specific>
    }
}

Regions: global, sky, face, subject, background, ground, foliage, water, building, clothing.
Colors: reds, oranges, yellows, greens, aquas, blues, purples, magentas.

Guidelines:
- Subtle adjustments (5-20) are usually better than extreme ones (40+).
- \"warm\" → temperature +10 to +20, oranges luminance boost
- \"film look\" → shadows +20 to +40, highlights -20 to -40, tone_curve blacks crush
- \"cinematic\" → teal shadows (blues h shift in shadows region), warm highlights, moderate contrast
- Skin tones: prefer vibrance over saturation, adjust oranges h/s/l in face/subject region
- whites/blacks control clipping points; use them for S-curve style effects
- dehaze > 0 cuts atmospheric haze; dehaze < 0 adds mist/atmosphere
- Output ONLY JSON, no explanation."""


def text_to_params(instruction, image_description=None):
    """Convert natural language to structured editing parameters.
    Returns a flat, region-keyed dict: {"global": {...}, "sky": {...}, ...}
    Each region's value is a valid params dict for renderer.render().
    """
    user_msg = f"Editing instruction: {instruction}"
    if image_description:
        user_msg = f"Image: {image_description}\n\n{user_msg}"

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}],
    )

    result = extract_json(response.content[0].text)
    if result is None:
        raise ValueError(f"Could not parse JSON: {response.content[0].text[:200]}")
    return result


print("text_to_params() ready")


text_to_params() ready


In [4]:
# Test with diverse prompts — each result is also saved as training data
test_prompts = [
    "make this photo warmer with lifted shadows, like a fujifilm classic chrome look",
    "reduce the highlights from the sun, make the shadows a bit brighter",
    "make the skin tones brighter and warmer, keep the background cool",
    "add film grain feel — slightly desaturate the greens, crush the blacks",
    "night street scene: moodier with crushed blacks and warm highlights",
    "lift shadows, compress highlights, teal shift in shadows for cinematic look",
]

print("Testing text-to-params conversion:\n")
all_results = []

for prompt in test_prompts:
    print(f'Prompt: "{prompt}"')
    try:
        params = text_to_params(prompt)
        print(f"  → {json.dumps(params, indent=2)}")
        all_results.append({"prompt": prompt, "params": params})
    except Exception as e:
        print(f"  Error: {e}")
    print()

# Save as seed training data for Pipeline 06
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
with open(f'{CHECKPOINTS_DIR}/text_to_params_examples.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"Saved {len(all_results)} prompt-parameter pairs")

Testing text-to-params conversion:

Prompt: "make this photo warmer with lifted shadows, like a fujifilm classic chrome look"
  → {
  "global": {
    "temperature": 15,
    "shadows": 35,
    "highlights": -15,
    "contrast": 10,
    "vibrance": -10,
    "saturation": -15,
    "tone_curve": {
      "blacks": 10,
      "shadows": 25,
      "highlights": -10
    },
    "color_mixer": {
      "oranges": {
        "h": -5,
        "s": -10,
        "l": 5
      },
      "yellows": {
        "s": -15,
        "l": 10
      },
      "greens": {
        "h": 10,
        "s": -20
      },
      "blues": {
        "h": 5,
        "s": -10
      }
    }
  }
}

Prompt: "reduce the highlights from the sun, make the shadows a bit brighter"
  → {
  "global": {
    "highlights": -40,
    "shadows": 25
  }
}

Prompt: "make the skin tones brighter and warmer, keep the background cool"
  → {
  "global": {
    "temperature": 15
  },
  "face": {
    "shadows": 25,
    "vibrance": 15,
    "color_mixer": {

## 2. Visual Test: Full Pipeline

Text prompt → Claude → JSON params → DeterministicRenderer → pixel transforms → output image.

We compare against your actual Lightroom edits to see how close the pipeline gets.

In [5]:
# Full pipeline test: text → params → render → compare with your Lightroom edit.
# With the flat format, params['global'] contains all adjustments for the full image
# (including tone_curve and color_mixer nested inside). Regional params (sky, face, etc.)
# require SAM segmentation — available in NB04+. Here we only apply the global layer.
renderer = DeterministicRenderer()
pairs = pair_files()
print(f"Loaded {len(pairs)} matched pairs")

test_cases = [
    {"idx": 0,                   "prompt": "apply warm cinematic style with lifted shadows and fujifilm feel"},
    {"idx": len(pairs)//4,       "prompt": "moodier with darker shadows and warm highlights, film grain"},
    {"idx": len(pairs)//2,       "prompt": "brighten the face, warm skin tones, cooler background"},
    {"idx": 3*len(pairs)//4,     "prompt": "reduce highlights, lift shadows, teal shift in shadows"},
]

fig, axes = plt.subplots(len(test_cases), 3, figsize=(18, 6 * len(test_cases)))

for row, test in enumerate(test_cases):
    raw_f, edited_f = pairs[test['idx']]
    raw_img = Image.open(os.path.join(RAW_DIR, raw_f)).convert('RGB')
    raw_np = np.array(raw_img).astype(np.float32) / 255.0

    params = text_to_params(test['prompt'])
    # Flat format: params['global'] holds all adjustments including nested tone_curve/color_mixer.
    # Regional keys (sky, face, etc.) are skipped here — they need SAM masks from NB04.
    global_params = params.get('global', {})
    result_np = renderer.render(raw_np, global_params, strength=1.0)
    result_img = Image.fromarray((result_np * 255).astype(np.uint8))

    axes[row, 0].imshow(raw_img)
    axes[row, 0].set_title(f'Raw: {raw_f}', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(result_img)
    axes[row, 1].set_title(f'Pipeline (global): "{test["prompt"][:40]}..."', fontsize=9)
    axes[row, 1].axis('off')

    your_edit = Image.open(os.path.join(EDITED_DIR, edited_f)).convert('RGB')
    axes[row, 2].imshow(your_edit)
    axes[row, 2].set_title('Your Lightroom Edit', fontsize=9)
    axes[row, 2].axis('off')

plt.suptitle('Text → Params (flat format) → Global Render vs Your Edits', fontsize=14)
plt.tight_layout()
plt.show()


Output hidden; open in https://colab.research.google.com to view.

In [6]:
# Analyze existing raw→edited pairs to understand your editing patterns.
# Each analysis becomes a (description, params) training pair.
import base64
from io import BytesIO

def analyze_edit(raw_path, edited_path):
    """Use Claude to estimate what Lightroom params were applied."""
    raw_img = Image.open(raw_path).convert('RGB')
    edited_img = Image.open(edited_path).convert('RGB')
    raw_b64 = image_to_base64(raw_img)
    edited_b64 = image_to_base64(edited_img)

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=2000,
        system="""You are an expert photo editor analyzing before/after pairs.
Estimate the Lightroom adjustments applied. Respond with ONLY valid JSON.

Required fields:
- "style_description": natural language summary of the edit
- "global": object with parameter names and numeric values (only non-zero)
- "tone_curve": object with {blacks, shadows, midtones, highlights, whites} if relevant
- "color_mixer": object with color names and {h, s, l} shifts if relevant

Use standard Lightroom parameter names and ranges. Output ONLY JSON.""",
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": raw_b64}},
                {"type": "text", "text": "RAW (before) image."},
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": edited_b64}},
                {"type": "text", "text": "EDITED (after) image. Output ONLY JSON with estimated parameters."},
            ]
        }],
    )

    result = extract_json(response.content[0].text)
    if result is None:
        raise ValueError(f"Could not parse: {response.content[0].text[:200]}")
    return result


# Analyze 6 evenly-spaced pairs
pairs = pair_files()
sample_indices = np.linspace(0, len(pairs)-1, 6, dtype=int)

analyses = []
for idx in sample_indices:
    raw_f, edited_f = pairs[idx]
    print(f"Analyzing {raw_f} → {edited_f}...", end=" ")
    try:
        analysis = analyze_edit(
            os.path.join(RAW_DIR, raw_f),
            os.path.join(EDITED_DIR, edited_f),
        )
        analyses.append({"raw": raw_f, "edited": edited_f, "analysis": analysis})
        print(f"[{analysis.get('style_description', '')[:50]}]")
    except Exception as e:
        print(f"Error: {e}")

with open(f'{CHECKPOINTS_DIR}/edit_analyses.json', 'w') as f:
    json.dump(analyses, f, indent=2)
print(f"\nSaved {len(analyses)} edit analyses")

Analyzing raw_001.jpg → edited_01.jpg... [Subtle warm enhancement with increased contrast an]
Analyzing raw_058.jpg → edited_58.jpg... Error: Could not parse: ```json
{
  "style_description": "Subtle film-like processing with slight warmth adjustment, reduced contrast for a softer look, and gentle highlight recovery to retain sky detail",
  "global": {
    
Analyzing raw_115.jpg → edited_115.jpg... [Subtle contrast and clarity enhancement with sligh]
Analyzing raw_172.jpg → edited_172.jpg... [Bright, airy edit with lifted shadows and enhanced]
Analyzing raw_229.jpg → edited_229.jpg... [Subtle warm film-like enhancement with lifted shad]
Analyzing raw_287.jpg → edited_287.jpg... [Subtle portrait enhancement with warmer tones, lif]

Saved 5 edit analyses


In [7]:
# Save pipeline config — documents what this pipeline can do
pipeline_config = {
    'version': '2.0',
    'architecture': 'hybrid_text_to_params',
    'renderer': 'DeterministicRenderer (HSV tone curves + 8-channel HSL mixer + basic params)',
    'text_interpreter': 'Claude Sonnet 4 (via API)',
    'shared_module': 'src/shared.py',
    'schema': EDIT_SCHEMA,
    'num_training_pairs': len(pairs),
}

with open(f'{CHECKPOINTS_DIR}/pipeline_config.json', 'w') as f:
    json.dump(pipeline_config, f, indent=2)

print("Pipeline 01 complete.")
print("  → text_to_params_examples.json (seed training data)")
print("  → edit_analyses.json (pair analyses)")
print("  → pipeline_config.json")
print("\nNext: Pipeline 02 (Style Profiling + Region Detection)")

Pipeline 01 complete.
  → text_to_params_examples.json (seed training data)
  → edit_analyses.json (pair analyses)
  → pipeline_config.json

Next: Pipeline 02 (Style Profiling + Region Detection)
